# Food.com Baseline Evaluation

Dieses Notebook führt unsere erste saubere Offline-Evaluation auf dem Food.com-Dataset aus. Wir behandeln das Problem als Top-N-Recommendation mit positivem implizitem Feedback.

Modellierungsannahme für diese erste Version:

- positives Feedback: `rating >= 4`
- nur User mit mindestens 2 positiven Interaktionen werden evaluiert
- Split: strict leave-last-out nach Zeitstempel

### Inhaltsübersicht:
1. Imports & Setup
2. Datapipeline & Preprocessing
3. Model Training
4. Offline Evaluation  
5. Retrieval Evaluation & ANN Analysis 
6. Learning-to-Rank (LTR)  
7. Slice & Hybrid Analysis  
8. Diagnostics & Explainability  
9. Experiment Logging



### To Do:
- Markdowns
- hübschere Tabellen
- ein paar Plots


## 1. Imports & Setup

Wir laden die Evaluationsfunktionen und die Modelle, setzen Seeds für Reproduzierbarkeit und definieren zentrale Laufzeit-Schalter.
Für diesen finalen Run sind alle Modelle aktiviert (`RUN_KNN_ON_FULL_DATA`, `RUN_GRAPH_PPR` und `RUN_CONTENT_BASED`).

Um RAM-Probleme zu vermeiden, wird das kNN-Modell intern auf die Top 30.000 Items limitiert. Das Content-Based Modell nutzt nun den vollen Katalog (`CONTENT_CANDIDATE_ITEMS = None`), und die Auswertung läuft über alle CPU-Kerne (`joblib`), um die über 44.000 User effizient zu evaluieren.

In [1]:
import os
import sys
import csv
import random
import copy
from pathlib import Path
from datetime import datetime

import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from joblib import Parallel, delayed

cwd = Path.cwd().resolve()
project_root = cwd if (cwd / "evaluation").exists() else cwd.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print("=== PHASE 1: SETUP ===")
print("Current working directory:", cwd)
print("Project root used:", project_root)

from evaluation.split import leave_last_out_split
from evaluation.metrics import recall_at_k, ndcg_at_k, catalog_coverage_at_k, novelty_at_k, snips_recall_at_k
from evaluation.retrieval_metrics import summarize_retrieval_metrics, retrieval_latency_ms

from models.baselines import PopularityRecommender, RandomRecommender, TrendingRecommender
from models.content_based import ContentBasedRecommender, HybridMFContentRecommender
from models.knn import ItemItemKNN
from models.mf import BiasedMatrixFactorization, BPRMatrixFactorization
from models.ltr_ranker import PointwiseLTRRanker, PairwiseXGBRanker
from models.ann_retriever import ANNMatrixFactorizationRetriever
from models.sequential import MarkovRecommender
from models.graph_recommender import PersonalizedPageRankRecommender

np.random.seed(42)
random.seed(42)
K = 20

MAX_EVAL_USERS = None
EVAL_SAMPLE_RANDOM_STATE = 42
EVALUATE_WARM_ITEMS_ONLY = False

RUN_DEBIASED_BPR = True
DEBIASED_BPR_BETA = 0.15

RUN_GRAPH_PPR = True
MAX_RETRIEVAL_EVAL_USERS = None
MAX_LTR_USERS = None
MAX_LATENCY_USERS = 500

RUN_KNN_ON_FULL_DATA = True
RUN_CONTENT_BASED = True
CONTENT_CANDIDATE_ITEMS = None


=== PHASE 1: SETUP ===
Current working directory: C:\Users\pssol\PycharmProjects\recommended-systems\notebooks
Project root used: C:\Users\pssol\PycharmProjects\recommended-systems


## 2. Datapipeline & Preprocessing

Hier bauen wir den finalen Modellierungsdatensatz auf:

- Food.com-RAW-Dateien laden
- Ratings in positives implizites Feedback umwandeln
- nur evaluierbare User behalten
- Rezept-Metadaten joinen
- leave-last-out Split vorbereiten

In [2]:
print("=== PHASE 2: DATA PIPELINE ===")

interactions = pd.read_csv("../data/raw/RAW_interactions.csv")
recipes = pd.read_csv("../data/raw/RAW_recipes.csv")

interactions["date"] = pd.to_datetime(interactions["date"], errors="coerce")
recipes["submitted"] = pd.to_datetime(recipes["submitted"], errors="coerce")

# Positive implicit feedback: ratings 4 and 5.
positive_df = interactions[interactions["rating"] >= 4].copy()
positive_df = positive_df.rename(columns={"date": "date_time"})
positive_df = positive_df.sort_values(["user_id", "date_time"])

# Leave-last-out needs at least two positive interactions per user.
user_counts = positive_df["user_id"].value_counts()
valid_users = user_counts[user_counts >= 2].index
positive_df = positive_df[positive_df["user_id"].isin(valid_users)].copy()

recipes_model = recipes.rename(columns={"id": "recipe_id"}).copy()
df_model = positive_df.merge(recipes_model, on="recipe_id", how="inner")

print(f"Positive interactions after filtering: {len(df_model):,}")
print(f"Users for evaluation: {df_model['user_id'].nunique():,}")
print(f"Recipes in final dataset: {df_model['recipe_id'].nunique():,}")
print(f"Date range: {df_model['date_time'].min()} to {df_model['date_time'].max()}")

train_df, test_df = leave_last_out_split(df_model, user_col="user_id", time_col="date_time")

train_item_counts = train_df["recipe_id"].value_counts().to_dict()
total_train_interactions = len(train_df)
total_items = train_df["recipe_id"].nunique()
recipe_names = (
    recipes_model.drop_duplicates("recipe_id")
    .assign(name=lambda x: x["name"].fillna("Unknown Recipe"))
    .set_index("recipe_id")["name"]
    .to_dict()
)

print("\nPreview of final training data:")
display(train_df[["user_id", "recipe_id", "date_time", "rating", "name"]].head(5))

=== PHASE 2: DATA PIPELINE ===
Positive interactions after filtering: 875,778
Users for evaluation: 52,364
Recipes in final dataset: 206,817
Date range: 2000-01-25 00:00:00 to 2018-12-19 00:00:00
Sorting data chronologically to prevent time leakage...
Dropping users with tied final timestamps because a strict leave-last-out split is not identifiable: 8115 users
Splitting data: Leave-last-out...
Train Set: 707232 interactions
Test Set: 44249 interactions

Preview of final training data:


,user_id,recipe_id,date_time,rating,name
0,1533,17338,2002-02-19,4,zucchini and rice casserole
1,1533,24375,2002-04-23,5,costa rican stuffed tortilla
2,1533,10721,2002-05-02,5,orange yogurt cream
3,1533,23891,2002-05-06,5,parmesan fish in the oven
4,1533,24136,2002-05-20,5,fennel mashed potatoes


## 3. Model Training

Wir trainieren die Collaborative-Filtering-Baselines und zusätzlich einen Content-Based-Recommender.

Das **Content-Based Modell** wird auf dem gesamten Katalog trainiert, um maximale Coverage zu erzielen. Das **Item-Item kNN Modell** wird hingegen auf die Top 30.000 Rezepte limitiert, um Memory-Überläufe (RAM) bei der Matrixmultiplikation zu verhindern.

`Sequential Baseline: First-Order Markov`
Dieses Modell nutzt die Reihenfolge der Interaktionen und empfiehlt häufige Nachfolger des zuletzt gesehenen Rezepts.

`Graph-Based Baseline: Personalized PageRank`
Die Interaktionen werden als bipartiter User-Item-Graph interpretiert. Über random-walk-ähnliche Multi-Hop-Pfade werden Kandidaten gefunden, die strukturell nah am Zieluser liegen.

In [3]:
print("\n=== PHASE 3: BASELINE, SEQUENTIAL, GRAPH & RETRIEVAL MODEL TRAINING ===")


# ============================================================
# Popularity
# ============================================================

print("\n[START] Popularity Recommender")

pop_model = PopularityRecommender()
pop_model.fit(train_df, item_col="recipe_id")

print("[DONE] Popularity Recommender")


# ============================================================
# Trending
# ============================================================

print("\n[START] Trending Recommender")

trend_model = TrendingRecommender(days_window=180)
trend_model.fit(
    train_df,
    item_col="recipe_id",
    time_col="date_time"
)

print("[DONE] Trending Recommender")


# ============================================================
# Sequential Markov
# ============================================================

print("\n[START] Markov Sequential Recommender")

markov_model = MarkovRecommender()
markov_model.fit(
    train_df,
    user_col="user_id",
    item_col="recipe_id",
    time_col="date_time"
)

print("[DONE] Markov Sequential Recommender")


# ============================================================
# Graph PPR
# ============================================================

ppr_model = None

# Graph-based Personalized PageRank can be very expensive to train on large datasets like Food.com,
# so we make it optional and allow skipping it if desired.
if RUN_GRAPH_PPR:
    print("\n[START] Graph PPR Recommender")

    ppr_model = PersonalizedPageRankRecommender(
        alpha=0.15,
        max_steps=3,
        n_walks=100,
        random_state=42
    )

    ppr_model.fit(
        train_df,
        user_col="user_id",
        item_col="recipe_id"
    )

    print("[DONE] Graph PPR Recommender")
else:
    print("\n[SKIPPED] Graph PPR Recommender")


# ============================================================
# Random
# ============================================================

print("\n[START] Random Recommender")

rand_model = RandomRecommender()
rand_model.fit(train_df, item_col="recipe_id")

print("[DONE] Random Recommender")


# ============================================================
# Content-Based
# ============================================================

content_model = None

if RUN_CONTENT_BASED:

    print("\n[START] Content-Based Recommender")

    content_candidate_ids = (
        train_df["recipe_id"]
        .value_counts()[:CONTENT_CANDIDATE_ITEMS]
        .index
    )

    content_items_df = recipes_model[
        recipes_model["recipe_id"].isin(content_candidate_ids)
    ].copy()

    content_train_df = train_df[
        train_df["recipe_id"].isin(content_candidate_ids)
    ].copy()

    content_model = ContentBasedRecommender(
        text_cols=["name", "tags", "ingredients", "description"],
        metadata_cols=["minutes", "n_steps", "n_ingredients"],
        min_df=2,
        ngram_range=(1, 1),
        time_col="date_time",
        recency_decay=0.0,
    )

    print(
        f"Training Content-Based model on top "
        f"{len(content_candidate_ids):,} recipes by train popularity..."
    )

    content_model.fit(
        content_train_df,
        content_items_df,
        user_col="user_id",
        item_col="recipe_id"
    )

    print("[DONE] Content-Based Recommender")


# ============================================================
# Item-Item kNN
# ============================================================
knn_model = None

if RUN_KNN_ON_FULL_DATA:
    print("\n[START] Item-Item kNN")

    # --- RAM SAVER: Begrenze kNN auf die 30.000 beliebtesten Items ---
    # Das verhindert, dass die RAM-Auslastung bei der Matrix-Multiplikation 32GB sprengt.
    knn_candidate_ids = train_df["recipe_id"].value_counts().head(30000).index
    knn_train_df = train_df[train_df["recipe_id"].isin(knn_candidate_ids)].copy()

    knn_model = ItemItemKNN(
        k_neighbors=100,
        shrinkage=10
    )
    knn_model.fit(knn_train_df, item_col="recipe_id")
    print("[DONE] Item-Item kNN")


# ============================================================
# Biased MF
# ============================================================

print("\n[START] Biased Matrix Factorization")

mf_model = BiasedMatrixFactorization(
    k_factors=16,
    epochs=5
)

mf_model.fit(train_df, item_col="recipe_id")

print("[DONE] Biased Matrix Factorization")


# ============================================================
# BPR MF
# ============================================================

print("\n[START] BPR Matrix Factorization")

bpr_model = BPRMatrixFactorization(
    k_factors=32,
    epochs=5
)

bpr_model.fit(train_df, item_col="recipe_id")

print("[DONE] BPR Matrix Factorization")


bpr_debiased_model = None

if RUN_DEBIASED_BPR:
    print("\n[START] Debiased BPR Matrix Factorization")

    bpr_debiased_model = copy.deepcopy(bpr_model)
    bpr_debiased_model.popularity_penalty_beta = DEBIASED_BPR_BETA

    print(
        f"[DONE] Debiased BPR created from trained BPR "
        f"with beta={DEBIASED_BPR_BETA}"
    )

# ============================================================
# ANN Retrieval
# ============================================================

print("\n[START] ANN Retrieval Index")

print("Building ANN retrieval index from learned BPR item embeddings...")

ann_bpr_model = ANNMatrixFactorizationRetriever(
    backend="auto",
    normalize=True
)

ann_bpr_model.fit_from_mf_model(bpr_model)

print("[DONE] ANN Retrieval Index")


# ============================================================
# Hybrid
# ============================================================

hybrid_model = None

if content_model is not None:

    print("\n[START] Hybrid BPR + Content Recommender")

    hybrid_model = HybridMFContentRecommender(
        mf_model=bpr_model,
        content_model=content_model,
        alpha=0.9,
        adaptive=True,
        sparse_threshold=5
    )

    print("[DONE] Hybrid BPR + Content Recommender")


print("\n=== ALL ENABLED MODELS TRAINED AND READY FOR EVALUATION ===")




=== PHASE 3: BASELINE, SEQUENTIAL, GRAPH & RETRIEVAL MODEL TRAINING ===

[START] Popularity Recommender
Training Popularity Baseline...
Learned popularity ranking for 185226 unique recipes.
[DONE] Popularity Recommender

[START] Trending Recommender
Training Trending Baseline (last 180 days)...
Learned 795 trending recipes.
[DONE] Trending Recommender

[START] Markov Sequential Recommender
Training First-Order Markov Sequential Recommender...
Learned transitions for 180526 source recipes.
[DONE] Markov Sequential Recommender

[START] Graph PPR Recommender
Training Graph-Based Personalized PageRank Recommender (alpha=0.15, max_steps=3, n_walks=100)...
Graph contains 44,249 users, 185,226 items, and 707,232 user-item edges.
[DONE] Graph PPR Recommender

[START] Random Recommender
Training Random Baseline...
Random candidate universe contains 185226 recipes.
[DONE] Random Recommender

[START] Content-Based Recommender
Training Content-Based model on top 185,226 recipes by train popularit

## 4. Offline Evaluation

Jetzt evaluieren wir alle aktivierten Modelle auf dem strict temporal leave-last-out Split. Neben Recall@20, NDCG@20, Coverage@20 und Novelty@20 berechnen wir nun auch **SNIPS@20**. Diese Metrik gewichtet Treffer bei seltenen Rezepten höher als bei Mainstream-Rezepten und hilft uns, den Popularity Bias zu quantifizieren (Debiasing).

Um die Laufzeit auf dem riesigen Food.com-Datensatz und bei über 50.000 Usern praktikabel zu halten, wurde die Evaluierung mit `joblib` über alle verfügbaren CPU-Kerne parallelisiert.

In [4]:
print("\n=== PHASE 4: RANKING EVALUATION ===")

results = {
    "Popularity": {"ndcgs": [], "recalls": [], "novelties": [], "snips_recalls": [], "all_recs": []},
    "Trending (180d)": {"ndcgs": [], "recalls": [], "novelties": [], "snips_recalls": [], "all_recs": []},
    "Markov Sequential": {"ndcgs": [], "recalls": [], "novelties": [], "snips_recalls": [], "all_recs": []},
    "Graph PPR": {"ndcgs": [], "recalls": [], "novelties": [], "snips_recalls": [], "all_recs": []},
    "Random": {"ndcgs": [], "recalls": [], "novelties": [], "snips_recalls": [], "all_recs": []},
    "Content-Based": {"ndcgs": [], "recalls": [], "novelties": [], "snips_recalls": [], "all_recs": []},
    "Biased MF (SQ)": {"ndcgs": [], "recalls": [], "novelties": [], "snips_recalls": [], "all_recs": []},
    "BPR MF": {"ndcgs": [], "recalls": [], "novelties": [], "snips_recalls": [], "all_recs": []},
    "BPR MF Debiased": {"ndcgs": [], "recalls": [], "novelties": [], "snips_recalls": [], "all_recs": []},
    "ANN BPR Retrieval": {"ndcgs": [], "recalls": [], "novelties": [], "snips_recalls": [], "all_recs": []},
    "Hybrid BPR+Content": {"ndcgs": [], "recalls": [], "novelties": [], "snips_recalls": [], "all_recs": []}
}

models = {
    "Popularity": pop_model,
    "Trending (180d)": trend_model,
    "Markov Sequential": markov_model,
    "Graph PPR": ppr_model,
    "Random": rand_model,
    "Content-Based": content_model,
    "Biased MF (SQ)": mf_model,
    "BPR MF": bpr_model,
    "BPR MF Debiased": bpr_debiased_model,
    "ANN BPR Retrieval": ann_bpr_model,
    "Hybrid BPR+Content": hybrid_model
}

if ppr_model is None:
    results.pop("Graph PPR")
    models.pop("Graph PPR")

if bpr_debiased_model is None:
    results.pop("BPR MF Debiased")
    models.pop("BPR MF Debiased")
    
if ann_bpr_model is None:
    results.pop("ANN BPR Retrieval")
    models.pop("ANN BPR Retrieval")
    
if hybrid_model is None:
    results.pop("Hybrid BPR+Content")
    models.pop("Hybrid BPR+Content")

if content_model is None:
    results.pop("Content-Based")
    models.pop("Content-Based")

if knn_model is not None:
    results["Item-Item kNN"] = {"ndcgs": [], "recalls": [], "novelties": [], "all_recs": []}
    models["Item-Item kNN"] = knn_model

print("Preparing user histories...")
train_history = train_df.groupby("user_id")["recipe_id"].apply(set).to_dict()

# Map each test user to their true test item for easy lookup during evaluation.
# Not including cold-item users in warm-only evaluation
true_item_by_user = test_df.set_index("user_id")["recipe_id"].to_dict()

warm_test_users = [
    user for user, item in true_item_by_user.items()
    if item in train_item_counts
]

cold_test_users = [
    user for user, item in true_item_by_user.items()
    if item not in train_item_counts
]

if EVALUATE_WARM_ITEMS_ONLY:
    test_users = np.array(warm_test_users)
else:
    test_users = test_df["user_id"].unique()

if MAX_EVAL_USERS is not None and len(test_users) > MAX_EVAL_USERS:
    rng = np.random.default_rng(EVAL_SAMPLE_RANDOM_STATE)
    test_users = rng.choice(test_users, size=MAX_EVAL_USERS, replace=False)

print(f"Warm test users: {len(warm_test_users):,}")
print(f"Cold-item test users excluded from main eval: {len(cold_test_users):,}")
print(f"Evaluating {len(test_users):,} users. This can take a while...")

# Evaluate each model for each test user and store results
# Maximum 12000 users (instead of all 42k) to speed up evaluation, can be adjusted as needed (MAX_EVAL_USERS)
print(f"Evaluating {len(test_users):,} users using ALL CPU CORES. Please hold on...")

# Define the evaluation task for a single user
def evaluate_single_user(user):
    true_item = true_item_by_user[user]
    user_hist = train_history.get(user, set())

    user_results = {}
    for name, model in models.items():
        recs = model.recommend(user, user_history=user_hist, k=K)
        user_results[name] = {
            "all_recs": recs,
            "recall": recall_at_k(recs, [true_item], k=K),
            "ndcg": ndcg_at_k(recs, [true_item], k=K),
            "novelty": novelty_at_k(recs, train_item_counts, total_train_interactions, k=K)
        }
    return user_results

# Run the evaluation in parallel across all CPU cores
parallel_results = Parallel(n_jobs=-1, batch_size=100)(
    delayed(evaluate_single_user)(user) for user in test_users
)

# Merge the parallel results back into the main results dictionary
for user_res in parallel_results:
    for name in models.keys():
        results[name]["all_recs"].append(user_res[name]["all_recs"])
        results[name]["recalls"].append(user_res[name]["recall"])
        results[name]["ndcgs"].append(user_res[name]["ndcg"])
        results[name]["novelties"].append(user_res[name]["novelty"])

print("Parallel evaluation complete!")

# --- GLOBAL SNIPS CALCULATION ---
max_count = max(train_item_counts.values()) if train_item_counts else 1.0

for name in models.keys():
    snips_numerator = 0.0
    snips_denominator = 0.0

    for i, user in enumerate(test_users):
        true_item = true_item_by_user[user]
        count = train_item_counts.get(true_item, 0)
        # Propensity Proxy: sqrt(popularity), gecapped bei 0.05 für Stabilität
        p_i = max(0.05, np.sqrt(count / max_count))
        inv_p_i = 1.0 / p_i

        snips_denominator += inv_p_i
        if true_item in results[name]["all_recs"][i]:
            snips_numerator += inv_p_i

    results[name]["global_snips"] = snips_numerator / snips_denominator if snips_denominator > 0 else 0.0

print("\n--- FINAL MACRO-AVERAGED RESULTS TABLE ---")
print(f"{'Model':<20} | {'Recall@20':<10} | {'NDCG@20':<10} | {'Coverage@20':<12} | {'Novelty@20':<10} | {'SNIPS@20'}")
print("-" * 88)

for name in models.keys():
    cov = catalog_coverage_at_k(results[name]["all_recs"], total_items)
    mean_rec = np.mean(results[name]["recalls"])
    mean_ndcg = np.mean(results[name]["ndcgs"])
    mean_nov = np.mean(results[name]["novelties"])
    mean_snips = results[name]["global_snips"]
    print(f"{name:<20} | {mean_rec:.4f}     | {mean_ndcg:.4f}     | {cov:.4%} | {mean_nov:.4f}     | {mean_snips:.4f}")
    
if np.mean(results["Popularity"]["recalls"]) > 0.15:
    print("\nWARNING: Popularity baseline is very strong (>15% Recall). Check whether the split is too easy or too popular-item-heavy.")
else:
    print("\nSanity checks passed.")



=== PHASE 4: RANKING EVALUATION ===
Preparing user histories...
Warm test users: 39,380
Cold-item test users excluded from main eval: 4,869
Evaluating 44,249 users. This can take a while...
Evaluating 44,249 users using ALL CPU CORES. Please hold on...
Parallel evaluation complete!

--- FINAL MACRO-AVERAGED RESULTS TABLE ---
Model                | Recall@20  | NDCG@20    | Coverage@20  | Novelty@20 | SNIPS@20
----------------------------------------------------------------------------------------
Popularity           | 0.0312     | 0.0125     | 0.0367% | 10.3203     | 0.0036
Trending (180d)      | 0.0029     | 0.0018     | 0.0173% | 15.6854     | 0.0005
Markov Sequential    | 0.0173     | 0.0064     | 36.7961% | 12.9394     | 0.0038
Graph PPR            | 0.0064     | 0.0029     | 77.1463% | 15.6024     | 0.0029
Random               | 0.0002     | 0.0001     | 99.1351% | 18.3656     | 0.0001
Content-Based        | 0.0042     | 0.0018     | 54.8368% | 17.8774     | 0.0041
Biased MF (SQ

In [5]:
print("\n--- TWO-STAGE RECOMMENDER INTERPRETATION ---")

print("""
Stage 1: Candidate Generation (Retriever)
------------------------------------------
BPR MF:
- behavioral retrieval using collaborative filtering
- retrieves items from latent user-item interactions

ANN BPR Retrieval:
- approximate nearest-neighbor retrieval
- uses precomputed latent item embeddings
- simulates scalable production retrieval
      
Content-Based:
- semantic retrieval using recipe metadata
- retrieves items using ingredients, tags, and descriptions

Popularity:
- fallback candidate generator
- strong for sparse users and cold-start scenarios

Stage 2: Ranking
----------------
Hybrid BPR + Content:
- merges collaborative and content-based candidates
- reranks retrieved items using reciprocal-rank blending

Interpretation:
---------------
The ranker can only reorder what retrieval keeps alive.
If retrieval misses a relevant recipe, ranking cannot recover it.

This follows the standard two-stage recommender design:
cheap high-recall retrieval first, precise ranking second.
""")


--- TWO-STAGE RECOMMENDER INTERPRETATION ---

Stage 1: Candidate Generation (Retriever)
------------------------------------------
BPR MF:
- behavioral retrieval using collaborative filtering
- retrieves items from latent user-item interactions

ANN BPR Retrieval:
- approximate nearest-neighbor retrieval
- uses precomputed latent item embeddings
- simulates scalable production retrieval

Content-Based:
- semantic retrieval using recipe metadata
- retrieves items using ingredients, tags, and descriptions

Popularity:
- fallback candidate generator
- strong for sparse users and cold-start scenarios

Stage 2: Ranking
----------------
Hybrid BPR + Content:
- merges collaborative and content-based candidates
- reranks retrieved items using reciprocal-rank blending

Interpretation:
---------------
The ranker can only reorder what retrieval keeps alive.
If retrieval misses a relevant recipe, ranking cannot recover it.

This follows the standard two-stage recommender design:
cheap high-recall

## 5. Retrieval Evaluation & ANN Analysis

In diesem Abschnitt analysieren wir die Candidate-Generation
als erste Stufe eines modernen Two-Stage-Recommender-Systems.

Dabei wird untersucht:
- ob relevante Items im Candidate-Set überleben,
- wie sich verschiedene Retrieval-Budgets auswirken,
- wie breit die Retrieval-Modelle den Item-Katalog abdecken,
- und wie sich ANN-basiertes Retrieval gegenüber vollständigem Scoring verhält.

Evaluiert werden:
- Candidate Recall@K
- Candidate Coverage
- durchschnittliche Candidate-Set-Größe
- Duplicate Rates
- Retrieval Latency

Zusätzlich wird ein ANN-Retriever (Approximate Nearest Neighbors)
auf den gelernten BPR-Embeddings evaluiert.
Dabei werden Item-Embeddings offline indexiert und
ähnliche Kandidaten effizient im Embedding-Space gesucht.

In [6]:
# === RETRIEVAL EVALUATION & ANN ANALYSIS: Candidate Recall under different budgets ===
'''
This phase evaluates candidate generators independently from ranking quality.

ANN retrieval uses the learned BPR latent item vectors
and performs nearest-neighbor search in embedding space.

This follows the modern two-stage recommender architecture:
offline item indexing + online user retrieval.
'''

candidate_budgets = [50, 100, 300, 500]

retrieval_models = {
    "Popularity": pop_model,
    "Trending (180d)": trend_model,
    "Markov Sequential": markov_model,
    "Graph PPR": ppr_model,
    "Random": rand_model,
    "Content-Based": content_model,
    "Biased MF (SQ)": mf_model,
    "BPR MF": bpr_model,
    "BPR MF Debiased": bpr_debiased_model,
    "ANN BPR Retrieval": ann_bpr_model,
    "Hybrid BPR+Content": hybrid_model,
}

# Remove disabled models automatically
retrieval_models = {
    name: model
    for name, model in retrieval_models.items()
    if model is not None
}

true_items_by_user = test_df.set_index("user_id")["recipe_id"].to_dict()

retrieval_rows = []

retrieval_users = test_users

if MAX_RETRIEVAL_EVAL_USERS is not None and len(retrieval_users) > MAX_RETRIEVAL_EVAL_USERS:
    rng = np.random.default_rng(EVAL_SAMPLE_RANDOM_STATE)
    retrieval_users = rng.choice(
        retrieval_users,
        size=MAX_RETRIEVAL_EVAL_USERS,
        replace=False
    )

print(f"Evaluating retrieval on {len(retrieval_users):,} users.")

for model_name, model in retrieval_models.items():
    print(f"Evaluating retrieval for {model_name}...")

    max_budget = max(candidate_budgets)
    all_candidate_lists = []

    # For each user, get the candidate list from the retriever with the maximum budget
    for user in retrieval_users:
        user_hist = train_history.get(user, set())

        candidates = model.recommend(
            user_id=user,
            user_history=user_hist,
            k=max_budget
        )

        all_candidate_lists.append(candidates)

    for budget in candidate_budgets:
        metrics = summarize_retrieval_metrics(
            all_candidate_lists=all_candidate_lists,
            true_items_by_user=true_items_by_user,
            user_ids=retrieval_users,
            total_catalog_size=total_items,
            k=budget,
        )

        retrieval_rows.append({
            "model": model_name,
            "candidate_budget": budget,
            **metrics,
        })

retrieval_df = pd.DataFrame(retrieval_rows)

print("\n=== PHASE 5: RETRIEVAL EVALUATION RESULTS ===")
display(retrieval_df)

Evaluating retrieval on 44,249 users.
Evaluating retrieval for Popularity...
Evaluating retrieval for Trending (180d)...
Evaluating retrieval for Markov Sequential...
Evaluating retrieval for Graph PPR...
Evaluating retrieval for Random...
Evaluating retrieval for Content-Based...
Evaluating retrieval for Biased MF (SQ)...
Evaluating retrieval for BPR MF...
Evaluating retrieval for BPR MF Debiased...
Evaluating retrieval for ANN BPR Retrieval...
Evaluating retrieval for Hybrid BPR+Content...

=== PHASE 5: RETRIEVAL EVALUATION RESULTS ===


,model,candidate_budget,candidate_recall_at_50,candidate_coverage_at_50,avg_candidate_set_size_at_50,mean_duplicate_rate_at_50,candidate_recall_at_100,candidate_coverage_at_100,avg_candidate_set_size_at_100,mean_duplicate_rate_at_100,candidate_recall_at_300,candidate_coverage_at_300,avg_candidate_set_size_at_300,mean_duplicate_rate_at_300,candidate_recall_at_500,candidate_coverage_at_500,avg_candidate_set_size_at_500,mean_duplicate_rate_at_500
0,Popularity,50,0.057493,0.000756,50.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Popularity,100,NaN,NaN,NaN,NaN,0.081516,0.001404,100.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Popularity,300,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.137043,0.003736,300.0,0.0,NaN,NaN,NaN,NaN
3,Popularity,500,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.171574,0.005669,500.0,0.0
4,Trending (180d),50,0.007209,0.000405,50.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,Trending (180d),100,NaN,NaN,NaN,NaN,0.017673,0.000713,100.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,Trending (180d),300,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.023616,0.001965,300.0,0.0,NaN,NaN,NaN,NaN
7,Trending (180d),500,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.032475,0.003110,500.0,0.0
8,Markov Sequential,50,0.038916,0.421248,50.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,Markov Sequential,100,NaN,NaN,NaN,NaN,0.063775,0.445764,100.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [7]:
print("\n=== PHASE 5.1: ANN SERVING LATENCY CHECK ===")
"This checks the serving-style latency of the ANN/vector-index retriever on a small user sample."

if ann_bpr_model is not None:
    latency_users = list(test_users[:MAX_LATENCY_USERS])
    latencies = []

    for user in latency_users:
        user_hist = train_history.get(user, set())
        _, latency_ms = ann_bpr_model.recommend_with_latency(
            user_id=user,
            user_history=user_hist,
            k=300,
        )
        latencies.append(latency_ms)

    print(f"ANN backend: {ann_bpr_model.active_backend}")
    print(f"Mean latency@300: {np.mean(latencies):.3f} ms")
    print(f"Median latency@300: {np.median(latencies):.3f} ms")
    print(f"P95 latency@300: {np.percentile(latencies, 95):.3f} ms")
else:
    print("ANN retriever disabled.")



=== PHASE 5.1: ANN SERVING LATENCY CHECK ===
ANN backend: numpy
Mean latency@300: 0.598 ms
Median latency@300: 0.581 ms
P95 latency@300: 0.703 ms


In [8]:
print("\n=== PHASE 5.2: RETRIEVAL LATENCY CHECK ===")

sample_user = test_users[0]
sample_history = train_history.get(sample_user, set())

_, brute_force_latency = retrieval_latency_ms(
    bpr_model.recommend,
    sample_user,
    sample_history,
    k=300
)

_, ann_latency = retrieval_latency_ms(
    ann_bpr_model.recommend,
    sample_user,
    sample_history,
    k=300
)

print(f"BPR brute-force latency: {brute_force_latency:.2f} ms")
print(f"ANN retrieval latency: {ann_latency:.2f} ms")


=== PHASE 5.2: RETRIEVAL LATENCY CHECK ===
BPR brute-force latency: 3.76 ms
ANN retrieval latency: 1.10 ms


## 6. Learning-to-Rank (LTR)

In diesem Abschnitt erweitern wir das bestehende Two-Stage-Recommender-System um einen starken Pairwise Learning-to-Rank Ranker.

Dafür werden:
- Candidate-Sets aus dem Hybrid Retriever erzeugt (Top 100).
- Das echte positive Item (True Item) explizit injiziert, falls der Retriever es verfehlt hat.
- Alle unangeklickten Items aus dem Retriever-Set als "Hard Negatives" deklariert.
- Ein **XGBoost Ranker** (Pairwise Loss via `rank:ndcg`) trainiert, der die Kandidaten neu ordnet.

Der Ranker lernt dabei aus Cross-Features:
- BPR-Scores
- Content-Scores
- Popularity-Features
- User-Aktivität

In [9]:
print("\n=== PHASE 6.1: BUILDING LTR TRAINING TABLE ===")

ltr_rows = []

# For LTR training, we focus on a smaller sample of users to keep the dataset manageable.
sample_users = test_users[:MAX_LTR_USERS]

for user in sample_users:
    true_item = true_item_by_user[user]
    user_hist = train_history.get(user, set())

    # Get hard negatives from the retriever
    candidates = hybrid_model.recommend(user, user_history=user_hist, k=100)
    training_items = set(candidates)
    training_items.add(true_item)

    for item in training_items:
        label = int(item == true_item)

        ltr_rows.append({
            "user_id": user,
            "recipe_id": item,
            "label": label,
            "bpr_score": bpr_model.score(user, item),
            "content_score": content_model.score(user, item),
            "popularity_score": pop_model.popularity_scores.get(item, 0),
            "user_activity": len(user_hist),
        })

ltr_df = pd.DataFrame(ltr_rows)

print("LTR rows:", len(ltr_df))
display(ltr_df.head())


=== PHASE 6.1: BUILDING LTR TRAINING TABLE ===
LTR rows: 4466629


,user_id,recipe_id,label,bpr_score,content_score,popularity_score,user_activity
0,1533,27144,0,3.168904,0.038008,262,119
1,1533,43023,0,3.017521,0.103631,201,119
2,1533,4627,0,3.177066,0.057351,344,119
3,1533,70165,0,3.154278,0.080286,252,119
4,1533,52253,0,3.091508,0.093771,191,119


In [10]:
print("\n=== PHASE 6.2: TRAINING LTR RANKER ===")

feature_columns = [
    "bpr_score",
    "content_score",
    "popularity_score",
    "user_activity",
]

ltr_ranker = PairwiseXGBRanker()

ltr_ranker.fit(
    train_df=ltr_df,
    feature_columns=feature_columns,
    label_column="label"
)

print("LTR ranker trained.")


=== PHASE 6.2: TRAINING LTR RANKER ===
LTR ranker trained.


In [11]:
print("\n=== PHASE 6.3: EVALUATING LTR RANKER ===")

ltr_recalls = []
ltr_ndcgs = []

# We evaluate the LTR ranker on a sample of test users to keep evaluation time reasonable.
for user in test_users[:MAX_LTR_USERS]:

    true_item = true_item_by_user[user]

    user_hist = train_history.get(user, set())

    candidates = hybrid_model.recommend(
        user,
        user_history=user_hist,
        k=100
    )

    feature_rows = []

    for item in candidates:

        feature_rows.append({
            "recipe_id": item,

            "bpr_score": bpr_model.score(user, item),
            "content_score": content_model.score(user, item),
            "popularity_score": pop_model.popularity_scores.get(item, 0),
            "user_activity": len(user_hist),
        })

    feature_df = pd.DataFrame(feature_rows)

    reranked = ltr_ranker.rerank(
        feature_df,
        top_k=20
    )

    ltr_recalls.append(
        recall_at_k(reranked, [true_item], k=20)
    )

    ltr_ndcgs.append(
        ndcg_at_k(reranked, [true_item], k=20)
    )

print("\nLTR RESULTS")
print("Recall@20:", np.mean(ltr_recalls))
print("NDCG@20:", np.mean(ltr_ndcgs))


=== PHASE 6.3: EVALUATING LTR RANKER ===

LTR RESULTS
Recall@20: 0.023751949196592015
NDCG@20: 0.008752400315653682


## 7. Slice & Hybrid Analysis

Zusätzlich zur globalen Offline-Evaluation analysieren wir die Modellperformance
auf unterschiedlichen Nutzer- und Item-Gruppen.

Dabei untersuchen wir:
- Unterschiede zwischen Sparse-, Medium- und Heavy-Usern,
- das Verhalten der Modelle bei Cold Items,
- sowie den Einfluss verschiedener Hybrid-Gewichtungen (`alpha`).

Ziel dieser Analysen ist es,
Popularity Bias, Cold-Start-Probleme
und Trade-offs zwischen Relevanz und Coverage
besser zu verstehen.

In [12]:
print("\n--- 7.1 USER ACTIVITY SLICE ANALYSIS ---")

user_train_counts = train_df.groupby("user_id")["recipe_id"].count().to_dict()

def user_activity_group(user_id):
    n = user_train_counts.get(user_id, 0)
    if n <= 3:
        return "Sparse users (<=3)"
    elif n <= 10:
        return "Medium users (4-10)"
    else:
        return "Heavy users (>10)"

slice_rows = []

# Wir nutzen enumerate, um den exakten Index aus Phase 4 wiederzufinden!
for i, user in enumerate(test_users):
    group = user_activity_group(user)
    true_item = true_item_by_user[user]

    for name in models.keys():
        # RECYCLE die Recommendations von vorhin! Kein neues model.recommend()!
        recs = results[name]["all_recs"][i]

        slice_rows.append({
            "model": name,
            "slice": group,
            "recall": recall_at_k(recs, [true_item], k=K),
            "ndcg": ndcg_at_k(recs, [true_item], k=K),
        })

slice_df = pd.DataFrame(slice_rows)

slice_summary = (
    slice_df
    .groupby(["model", "slice"])
    .agg(
        users=("recall", "count"),
        recall_at_20=("recall", "mean"),
        ndcg_at_20=("ndcg", "mean")
    )
    .reset_index()
)

display(slice_summary)


--- 7.1 USER ACTIVITY SLICE ANALYSIS ---


,model,slice,users,recall_at_20,ndcg_at_20
0,ANN BPR Retrieval,Heavy users (>10),8569,0.001400,0.000408
1,ANN BPR Retrieval,Medium users (4-10),9716,0.002367,0.000692
2,ANN BPR Retrieval,Sparse users (<=3),25964,0.002542,0.000789
3,BPR MF,Heavy users (>10),8569,0.029758,0.011201
4,BPR MF,Medium users (4-10),9716,0.032009,0.011005
5,BPR MF,Sparse users (<=3),25964,0.031120,0.012056
6,BPR MF Debiased,Heavy users (>10),8569,0.025441,0.009370
7,BPR MF Debiased,Medium users (4-10),9716,0.030259,0.010179
8,BPR MF Debiased,Sparse users (<=3),25964,0.028424,0.009655
9,Biased MF (SQ),Heavy users (>10),8569,0.017622,0.006557


In [13]:
print("\n--- 7.2 COLD ITEM SLICE ANALYSIS ---")

cold_items = ContentBasedRecommender.cold_item_set(
    train_df,
    test_df,
    item_col="recipe_id"
)

print(f"Cold items in test set: {len(cold_items)}")

cold_rows = []

for i, user in enumerate(test_users):
    true_item = true_item_by_user[user]

    if true_item not in cold_items:
        continue

    for name in models.keys():
        recs = results[name]["all_recs"][i] # RECYCLE!

        cold_rows.append({
            "model": name,
            "recall": recall_at_k(recs, [true_item], k=K),
            "ndcg": ndcg_at_k(recs, [true_item], k=K),
        })

cold_df = pd.DataFrame(cold_rows)

if len(cold_df) == 0:
    print("No cold items found in test set under current split.")
else:
    cold_summary = (
        cold_df
        .groupby("model")
        .agg(
            users=("recall", "count"),
            recall_at_20=("recall", "mean"),
            ndcg_at_20=("ndcg", "mean")
        )
        .reset_index()
    )

    display(cold_summary)


--- 7.2 COLD ITEM SLICE ANALYSIS ---
Cold items in test set: 4633


,model,users,recall_at_20,ndcg_at_20
0,ANN BPR Retrieval,4869,0.0,0.0
1,BPR MF,4869,0.0,0.0
2,BPR MF Debiased,4869,0.0,0.0
3,Biased MF (SQ),4869,0.0,0.0
4,Content-Based,4869,0.0,0.0
5,Graph PPR,4869,0.0,0.0
6,Hybrid BPR+Content,4869,0.0,0.0
7,Item-Item kNN,4869,0.0,0.0
8,Markov Sequential,4869,0.0,0.0
9,Popularity,4869,0.0,0.0


In [14]:
print("\n--- 7.3 HYBRID FINAL ALPHA CHECK ---")

'''
sample_users = test_users[:1000]

Tested different alpha values for the Hybrid BPR+Content model 
- to see how the balance between collaborative and content-based retrieval affects performance. 

alpha_values = [0.3, 0.5, 0.7, 0.9]
alpha_rows = []

for alpha in alpha_values:
    temp_hybrid = HybridMFContentRecommender(
        mf_model=bpr_model,
        content_model=content_model,
        alpha=alpha,
        adaptive=True,
        sparse_threshold=5
    )

    recalls = []
    ndcgs = []
    all_recs = []

    for user in sample_users:
        true_item = test_df[test_df["user_id"] == user]["recipe_id"].iloc[0]
        user_hist = train_history.get(user, set())

        recs = temp_hybrid.recommend(user, user_history=user_hist, k=K)

        recalls.append(recall_at_k(recs, [true_item], k=K))
        ndcgs.append(ndcg_at_k(recs, [true_item], k=K))
        all_recs.append(recs)

    alpha_rows.append({
        "alpha": alpha,
        "recall_at_20": np.mean(recalls),
        "ndcg_at_20": np.mean(ndcgs),
        "coverage_at_20": catalog_coverage_at_k(all_recs, total_items)
    })

alpha_df = pd.DataFrame(alpha_rows)
display(alpha_df)
'''


--- 7.3 HYBRID FINAL ALPHA CHECK ---


'\nsample_users = test_users[:1000]\n\nTested different alpha values for the Hybrid BPR+Content model \n- to see how the balance between collaborative and content-based retrieval affects performance. \n\nalpha_values = [0.3, 0.5, 0.7, 0.9]\nalpha_rows = []\n\nfor alpha in alpha_values:\n    temp_hybrid = HybridMFContentRecommender(\n        mf_model=bpr_model,\n        content_model=content_model,\n        alpha=alpha,\n        adaptive=True,\n        sparse_threshold=5\n    )\n\n    recalls = []\n    ndcgs = []\n    all_recs = []\n\n    for user in sample_users:\n        true_item = test_df[test_df["user_id"] == user]["recipe_id"].iloc[0]\n        user_hist = train_history.get(user, set())\n\n        recs = temp_hybrid.recommend(user, user_history=user_hist, k=K)\n\n        recalls.append(recall_at_k(recs, [true_item], k=K))\n        ndcgs.append(ndcg_at_k(recs, [true_item], k=K))\n        all_recs.append(recs)\n\n    alpha_rows.append({\n        "alpha": alpha,\n        "recall_at_20

In [15]:
print("\n--- 7.4 HYBRID FINAL ---")

sample_users = test_users[:MAX_LTR_USERS]

# tested: [0.3, 0.5, 0.7, 0.9]--> best result: alpha = 0.9
# reason: best Recall@20 + NDCG@20 with strong coverage
# interpretation: Food.com benefits more from stronger collaborative filtering (BPR) than from stronger content weighting.

# Therefore, the final hybrid model uses:

alpha = 0.9

temp_hybrid = HybridMFContentRecommender(
    mf_model=bpr_model,
    content_model=content_model,
    alpha=alpha,
    adaptive=True,
    sparse_threshold=5
)

recalls = []
ndcgs = []
all_recs = []

for user in sample_users:
    true_item = true_item_by_user[user]
    user_hist = train_history.get(user, set())

    recs = temp_hybrid.recommend(user, user_history=user_hist, k=K)

    recalls.append(recall_at_k(recs, [true_item], k=K))
    ndcgs.append(ndcg_at_k(recs, [true_item], k=K))
    all_recs.append(recs)

alpha_df = pd.DataFrame([{
    "alpha": alpha,
    "recall_at_20": np.mean(recalls),
    "ndcg_at_20": np.mean(ndcgs),
    "coverage_at_20": catalog_coverage_at_k(all_recs, total_items)
}])

display(alpha_df)


--- 7.4 HYBRID FINAL ---


,alpha,recall_at_20,ndcg_at_20,coverage_at_20
0,0.9,0.020927,0.007481,0.465453


## 8. Diagnostics And Explainability

Neben den Metriken schauen wir auf kurze Diagnostics: Content-Based-Erklaerungen, optional kNN-Erklaerungen, BPR-Bias und BPR-Latent-Space-Nachbarn.

In [16]:
print("\n=== PHASE 8: DIAGNOSTICS & EXPLAINABILITY ===")

if content_model is not None:
    print("\n--- 8A. CONTENT-BASED EXPLAINABILITY ---")
    cb_users = [u for u in train_df["user_id"].value_counts().index if u in content_model.user_profiles]
    sample_user = cb_users[0]
    explained_cb_recs = content_model.recommend(
        sample_user,
        user_history=train_history.get(sample_user, set()),
        k=3,
        return_explanations=True
    )
    print(f"Generating content-based explanations for User {sample_user}:")
    for i, (rec_id, explanation) in enumerate(explained_cb_recs, start=1):
        recipe_name = recipe_names.get(rec_id, "Unknown Recipe")
        print(f"{i}. {recipe_name}\n   -> {explanation}")
else:
    print("\n--- 8A. CONTENT-BASED EXPLAINABILITY ---")
    print("Skipped because Content-Based was disabled.")

if knn_model is not None:
    print("\n--- 8B. kNN CONSTRUCTIVE EXPLAINABILITY ---")
    heavy_users = train_df["user_id"].value_counts()
    sample_user = heavy_users[heavy_users > 5].sample(1, random_state=42).index[0]
    print(f"Generating explanations for User {sample_user}:")
    explained_recs = knn_model.recommend(
        sample_user,
        user_history=train_history.get(sample_user, set()),
        k=3,
        return_explanations=True
    )
    for i, (rec_id, explanation) in enumerate(explained_recs, start=1):
        recipe_name = recipe_names.get(rec_id, "Unknown Recipe")
        print(f"{i}. {recipe_name}\n   -> {explanation}")
else:
    print("\n--- 8B. kNN CONSTRUCTIVE EXPLAINABILITY ---")
    print("Skipped because Item-Item kNN was disabled for the full Food.com run.")

print("\n--- 8C. BPR BIAS INSPECTION ---")
top_bias_indices = np.argsort(bpr_model.b_i)[-5:][::-1]
print("Top 5 Recipes by Learned Bias:")
for idx in top_bias_indices:
    raw_id = bpr_model.reverse_item_mapping[idx]
    print(f"   -> {recipe_names.get(raw_id, 'Unknown Recipe')} (Bias: {bpr_model.b_i[idx]:.4f})")

print("\n--- 8D. BPR LATENT SPACE INSPECTION ---")
anchor_id = train_df["recipe_id"].value_counts().index[0]
anchor_idx = bpr_model.item_mapping[anchor_id]
anchor_vector = bpr_model.Q[anchor_idx].reshape(1, -1)

similarities = cosine_similarity(anchor_vector, bpr_model.Q)[0]
nearest_indices = [idx for idx in np.argsort(similarities)[::-1] if idx != anchor_idx][:5]

print(f"Anchor recipe: {recipe_names.get(anchor_id, 'Unknown Recipe')}")
print("Nearest latent neighbors:")
for idx in nearest_indices:
    raw_id = bpr_model.reverse_item_mapping[idx]
    print(f"   -> {recipe_names.get(raw_id, 'Unknown Recipe')} (Latent Sim: {similarities[idx]:.4f})")



=== PHASE 8: DIAGNOSTICS & EXPLAINABILITY ===

--- 8A. CONTENT-BASED EXPLAINABILITY ---
Generating content-based explanations for User 37449:
1. cucumber salad marinade
   -> Recommended because it overlaps on: tags: low, tags: or, tags: less.
2. baked tomato
   -> Recommended because it overlaps on: tags: low, tags: or, tags: fat.
3. easy pasta with bell pepper and tomatoes
   -> Recommended because it overlaps on: tags: low, description: the, tags: fat.

--- 8B. kNN CONSTRUCTIVE EXPLAINABILITY ---
Generating explanations for User 139991:
1. mashed red potatoes with garlic and parmesan
   -> Recommended because it is similar to a previously liked recipe (recipe_id=68955).
2. jo mama s world famous spaghetti
   -> Recommended because it is similar to a previously liked recipe (recipe_id=39087).
3. oven fried chicken chimichangas
   -> Recommended because it is similar to a previously liked recipe (recipe_id=39087).

--- 8C. BPR BIAS INSPECTION ---
Top 5 Recipes by Learned Bias:
   -> 

## 9. Experiment Logging

Zum Schluss schreiben wir alle aktivierten Modelllaeufe in `runs/runs.csv`, damit wir spaeter verschiedene Varianten vergleichen koennen.

In [17]:
print("\n=== PHASE 9: EXPERIMENT LOGGING ===")

runs_file = "../runs/runs.csv"
os.makedirs("../runs", exist_ok=True)

run_rows = []
for name, model in models.items():
    coverage = catalog_coverage_at_k(results[name]["all_recs"], total_items)
    hyperparams = ""
    if name == "BPR MF":
        hyperparams = f"k_factors={bpr_model.k}, epochs={bpr_model.epochs}, reg={bpr_model.reg}"
    elif name == "BPR MF Debiased":
        hyperparams = f"source=BPR MF, beta={DEBIASED_BPR_BETA}, k_factors={bpr_debiased_model.k}, epochs={bpr_debiased_model.epochs}, reg={bpr_debiased_model.reg}"
    elif name == "Biased MF (SQ)":
        hyperparams = f"k_factors={mf_model.k}, epochs={mf_model.epochs}, reg={mf_model.reg}"
    elif name == "ANN BPR Retrieval":
        hyperparams = f"source=BPR MF, backend={ann_bpr_model.active_backend}, include_bias={ann_bpr_model.include_item_bias}, overfetch={ann_bpr_model.overfetch_factor}"
    elif name == "Content-Based":
        hyperparams = f"top_items={CONTENT_CANDIDATE_ITEMS}, text=name/tags/ingredients/description, min_df={content_model.min_df}"
    elif name == "Trending (180d)":
        hyperparams = f"days_window={trend_model.days_window}"

    run_rows.append({
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "dataset": "Food.com_RAW_rating_ge_4_strict_split",
        "model": name,
        "k_evaluated": K,
        "hyperparams": hyperparams,
        "recall": round(np.mean(results[name]["recalls"]), 4),
        "ndcg": round(np.mean(results[name]["ndcgs"]), 4),
        "coverage": round(coverage, 4)
    })

file_exists = os.path.isfile(runs_file)
with open(runs_file, mode="a", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=run_rows[0].keys())
    if not file_exists:
        writer.writeheader()
    writer.writerows(run_rows)

print(f"Logged {len(run_rows)} model runs to {runs_file}!")
print("Notebook complete. Ready for comparison with future variants.")



=== PHASE 9: EXPERIMENT LOGGING ===
Logged 12 model runs to ../runs/runs.csv!
Notebook complete. Ready for comparison with future variants.
